# MedNorm-VI S1 - Mention First-Run Smoke Readiness

Colab Pro GPU notebook for a bounded S1 mention-extraction smoke. This is `SMOKE_ONLY`, not full training, organizer inference, or packaging.

## 1. Fixed Paths And Smoke Gate

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = "https://github.com/vquclinh/MedNorm-VI.git"
REPO_REF = "main"

if "MEDNORM_DRIVE_ROOT" in os.environ:
    DRIVE_ROOT = Path(os.environ["MEDNORM_DRIVE_ROOT"])
if "MEDNORM_REPO_DIR" in os.environ:
    REPO_DIR = Path(os.environ["MEDNORM_REPO_DIR"])
if "MEDNORM_REPO_URL" in os.environ:
    REPO_URL = os.environ["MEDNORM_REPO_URL"]
if "MEDNORM_REPO_REF" in os.environ:
    REPO_REF = os.environ["MEDNORM_REPO_REF"]

CORPUS_DIR = (
    DRIVE_ROOT
    / "data"
    / "derived"
    / "training_corpora"
    / "mednorm_vi_training_v1"
)
OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoint"
TRAINING_MANIFEST_PATH = OUTPUT_DIR / "training_manifest.json"
SMOKE_CONFIG_REL = Path("configs/training/s1_mention_first_run_smoke.yaml")

FULL_TRAINING_ENABLED = False
CONFIRM_FULL_TRAINING = ""
if FULL_TRAINING_ENABLED or CONFIRM_FULL_TRAINING:
    raise SystemExit("This notebook is SMOKE_ONLY; use the separate full S1 workflow.")

SEED = 20260723
print(json.dumps({
    "mode": "SMOKE_ONLY",
    "drive_root": str(DRIVE_ROOT),
    "repo_dir": str(REPO_DIR),
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "corpus_dir": str(CORPUS_DIR),
    "output_dir": str(OUTPUT_DIR),
}, indent=2, sort_keys=True))


## 2. Runtime And Drive Mount

In [ ]:
IN_COLAB = "google.colab" in sys.modules
runtime_report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
}
print(json.dumps(runtime_report, indent=2, sort_keys=True))
assert IN_COLAB, "S1 smoke must run on Google Colab Pro with a GPU; local execution is intentionally blocked."

drive_module = importlib.import_module("google.colab.drive")
drive_module.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Repository Clone And Commit Provenance

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    "git",
    "clone",
    "--branch",
    REPO_REF,
    "--single-branch",
    REPO_URL,
    str(REPO_DIR),
], check=True)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert len(RESOLVED_COMMIT) == 40 and all(c in "0123456789abcdef" for c in RESOLVED_COMMIT)
assert (REPO_DIR / "src" / "mednorm_vi").is_dir(), "cloned repo is missing src/mednorm_vi"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyYAML>=6.0"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
print(json.dumps({"resolved_commit": RESOLVED_COMMIT, "src_verified": True}, indent=2, sort_keys=True))


## 4. Corpus Gate Before Model Acquisition

In [ ]:
from mednorm_vi.model_registry.registry import load_registry, validate_profile_budget
from mednorm_vi.training.phobert_alignment import (
    ALIGNMENT_BACKEND,
    PARTIAL_TRUNCATION_POLICY,
    SUBTOKEN_SUPERVISION_POLICY,
    AlignmentError,
    describe_backend,
    map_segmented_words,
    segmented_text_to_words,
    verify_tokenizer_equivalence,
)
from mednorm_vi.training.s1_mention_smoke import (
    ENTITY_TYPE_ORDER,
    encode_mention_example_slow,
    expected_corpus_from_config,
    load_coverage,
    load_smoke_config,
    pad_encoded_features,
    preparation_digest,
    select_deterministic_examples,
    sha256_file,
    smoke_limits_from_config,
    verify_governed_corpus,
    verify_vietmed_train_only_masks,
)

smoke_config = load_smoke_config(REPO_DIR / SMOKE_CONFIG_REL)
expected_corpus = expected_corpus_from_config(smoke_config)
limits = smoke_limits_from_config(smoke_config)
print(json.dumps({
    "max_train_examples": limits.max_train_examples,
    "max_validation_examples": limits.max_validation_examples,
    "max_train_batches": limits.max_train_batches,
    "max_validation_batches": limits.max_validation_batches,
    "max_optimizer_steps": limits.max_optimizer_steps,
    "batch_size": limits.batch_size,
    "max_sequence_length": limits.max_sequence_length,
}, indent=2, sort_keys=True))

corpus_report = verify_governed_corpus(CORPUS_DIR, expected_corpus)
print(json.dumps(corpus_report, indent=2, sort_keys=True))


## 5. Registry, Parameter Budget, And Smoke Model Source

In [ ]:
model_cfg = smoke_config["model"]
roles = load_registry(REPO_DIR / "configs" / "model_registry" / "models_v1.yaml")
role = next(r for r in roles if r.model_id == model_cfg["registry_model_id"])
profile_budget = validate_profile_budget(roles, profile="full")
assert profile_budget.within_9b, "full model profile exceeds the 9B budget"
MODEL_ID = str(model_cfg["hf_model_id"])
MODEL_REVISION = str(model_cfg["revision"])
model_registry_report = {
    "registry_model_id": role.model_id,
    "role": role.role,
    "approved_model_source": model_cfg["approved_source"],
    "hf_model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION,
    "registry_base_parameters": role.base_parameter_count,
    "registry_adapter_parameters": role.adapter_parameter_count,
    "full_profile_base_parameters": profile_budget.base_parameters,
    "full_profile_adapter_parameters": profile_budget.adapter_parameters,
    "full_profile_total_parameters": profile_budget.total_parameters,
    "full_profile_within_9b": profile_budget.within_9b,
}
print(json.dumps(model_registry_report, indent=2, sort_keys=True))


## 6. Deterministic Smoke Subsets And Loss Masks

In [ ]:
coverage_by_source = load_coverage(CORPUS_DIR)
train_path = CORPUS_DIR / "splits" / "train.jsonl"
validation_path = CORPUS_DIR / "splits" / "validation.jsonl"
vietmed_train_examples = select_deterministic_examples(
    train_path,
    limit=1,
    seed=SEED,
    require_entities=True,
    source_dataset="vietmed_ner",
)
general_train_examples = select_deterministic_examples(
    train_path,
    limit=limits.max_train_examples,
    seed=SEED,
    require_entities=True,
)
seen_ids = set()
train_examples = []
for row in [*vietmed_train_examples, *general_train_examples]:
    if row["example_id"] not in seen_ids:
        train_examples.append(row)
        seen_ids.add(row["example_id"])
train_examples = train_examples[:limits.max_train_examples]
validation_examples = select_deterministic_examples(
    validation_path,
    limit=limits.max_validation_examples,
    seed=SEED,
    require_entities=True,
)
mask_report = verify_vietmed_train_only_masks(train_examples, coverage_by_source)
assert mask_report["vietmed_examples"] >= 1
assert mask_report["vietmed_entities"] >= 1
subset_report = {
    "train_examples": len(train_examples),
    "validation_examples": len(validation_examples),
    "vietmed_mask_report": mask_report,
    "train_split_sha256": sha256_file(train_path),
}
print(json.dumps(subset_report, indent=2, sort_keys=True))


## 7. Pinned Dependencies And GPU Check

In [ ]:
PIP_PACKAGES = [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "sentencepiece==0.2.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)

torch = importlib.import_module("torch")
transformers_module = importlib.import_module("transformers")
AutoModel = transformers_module.AutoModel
AutoTokenizer = transformers_module.AutoTokenizer

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "S1 smoke requires a Colab GPU runtime."
device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
runtime_report.update({
    "torch": torch.__version__,
    "cuda_available": True,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_vram_gb": round(props.total_memory / 1e9, 2),
    "pip_packages": PIP_PACKAGES,
})
print(json.dumps(runtime_report, indent=2, sort_keys=True))


## 8. Model And Tokenizer Acquisition Under Drive Cache

## Word segmentation contract (ViHealthBERT-Word expects segmented input)

ViHealthBERT-Word was pretrained on RDRSegmenter-segmented Vietnamese. Segmentation
resources are acquired **in Colab only** and cached under `DRIVE_ROOT/model_cache/vncorenlp`.
Their identity and hashes are recorded in the training manifest. If resources are missing
or their hashes do not match, the run fails fast rather than silently mis-segmenting.

`SEGMENTER_MODE = "whitespace_fallback"` is an explicitly recorded degraded mode for a
bounded smoke: it segments on whitespace only. It must never be used for real training.


In [ ]:
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"
# Production S1 smoke REQUIRES VnCoreNLP RDRSegmenter. whitespace_fallback is an
# explicit opt-in diagnostic mode only; it never activates automatically.
SEGMENTER_MODE = os.environ.get("MEDNORM_SEGMENTER_MODE", "vncorenlp")
assert SEGMENTER_MODE in ("vncorenlp", "whitespace_fallback"), SEGMENTER_MODE
DEGRADED_FALLBACK = SEGMENTER_MODE == "whitespace_fallback"

segmenter_report = {
    "segmenter_mode": SEGMENTER_MODE,
    "word_segmenter": "",
    "word_segmenter_version": "",
    "word_segmenter_resource_hashes": {},
    "degraded_fallback": DEGRADED_FALLBACK,
    "resource_dir": str(VNCORENLP_DIR),
    "acquisition_source": "",
}

if SEGMENTER_MODE == "vncorenlp":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py_vncorenlp==0.1.4"], check=True)
    VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
    py_vncorenlp = importlib.import_module("py_vncorenlp")
    if not any(VNCORENLP_DIR.glob("*.jar")):
        py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
    _resources = sorted(
        [q for q in VNCORENLP_DIR.rglob("*") if q.is_file()], key=lambda q: q.name)
    assert _resources, "VnCoreNLP resources missing after acquisition (fail fast)"
    _hashes = {q.name: sha256_file(q) for q in _resources if q.suffix in (".jar", ".xz", ".txt")}
    assert _hashes, "VnCoreNLP resource hashes are empty (broken installation; fail fast)"
    _rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))
    segmenter_report.update({
        "word_segmenter": "VnCoreNLP RDRSegmenter",
        "word_segmenter_version": "py_vncorenlp==0.1.4",
        "word_segmenter_resource_hashes": _hashes,
        "acquisition_source": "py_vncorenlp.download_model",
    })

    def segment_example_text(text):
        segments = _rdr.word_segment(text)
        assert segments, "RDRSegmenter returned no segments (fail fast)"
        return " ".join(segments)
else:
    print("=" * 78)
    print("!! DEGRADED MODE: whitespace_fallback is NOT the production S1 path.")
    print("!! Word segmentation does not match ViHealthBERT-Word pretraining.")
    print("!! This run cannot be classified as a successful production-path S1 smoke.")
    print("=" * 78)
    segmenter_report.update({
        "word_segmenter": "whitespace (degraded diagnostics only)",
        "word_segmenter_version": "builtin-whitespace",
        "acquisition_source": "none (degraded diagnostics mode)",
    })

    def segment_example_text(text):
        return " ".join(text.split())

PRODUCTION_SEGMENTATION = (
    segmenter_report["segmenter_mode"] == "vncorenlp"
    and segmenter_report["word_segmenter"] == "VnCoreNLP RDRSegmenter"
    and segmenter_report["degraded_fallback"] is False
    and bool(segmenter_report["word_segmenter_resource_hashes"])
)
print(json.dumps({k: v for k, v in segmenter_report.items()
                  if k != "word_segmenter_resource_hashes"}, indent=2, sort_keys=True))
print("resource_hash_count", len(segmenter_report["word_segmenter_resource_hashes"]))
print("production_segmentation", PRODUCTION_SEGMENTATION)


In [ ]:
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(MODEL_CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ViHealthBERT-Word declares tokenizer_class = PhobertTokenizer, which has NO fast
# implementation: requesting a fast tokenizer silently returns the slow one and
# tokenizer.is_fast stays False. Load it honestly as slow and align manually.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
    use_fast=False,
)
tokenizer_report = describe_backend(tokenizer)
tokenizer_report.update({
    "requested_revision": MODEL_REVISION,
    "pad_token_id": tokenizer.pad_token_id,
    "cls_token_id": getattr(tokenizer, "cls_token_id", None),
    "sep_token_id": getattr(tokenizer, "sep_token_id", None),
    "unk_token_id": getattr(tokenizer, "unk_token_id", None),
})
assert tokenizer_report["tokenizer_is_fast"] is False, (
    "expected the slow PhobertTokenizer; a fast tokenizer would change the alignment contract")
print(json.dumps(tokenizer_report, indent=2, sort_keys=True))


## Alignment preflight (runs BEFORE any model weights are downloaded)

Verifies the slow-tokenizer character alignment on the selected smoke examples:
word->original-span mapping, BPE subtoken alignment, label/mask dimensions, and the
truncation/boundary policies. Only aggregate diagnostics are printed.


## Tokenizer equivalence preflight (before any model weights are downloaded)

The alignment backend tokenizes **word by word** to attach character spans. This check
re-tokenizes the FULL segmented sentence with the real `PhobertTokenizer`
(`add_special_tokens=False`) and requires the token IDs to match the concatenated
per-word IDs exactly, for **every** selected smoke example. A mismatch fails the run
before `AutoModel.from_pretrained`.


In [ ]:
tokenizer_equivalence = {
    "tokenizer_equivalence_checked": True,
    "tokenizer_equivalence_examples": 0,
    "tokenizer_equivalence_failures": 0,
}
_equivalence_errors = []
for row in train_examples + validation_examples:
    try:
        _words = map_segmented_words(
            row["text"], segmented_text_to_words(segment_example_text(row["text"])))
    except AlignmentError:
        continue  # unalignable examples are counted by the alignment preflight
    try:
        verify_tokenizer_equivalence(_words, tokenizer)
        tokenizer_equivalence["tokenizer_equivalence_examples"] += 1
    except AlignmentError as exc:
        tokenizer_equivalence["tokenizer_equivalence_failures"] += 1
        _equivalence_errors.append(str(exc))

assert tokenizer_equivalence["tokenizer_equivalence_failures"] == 0, (
    "per-word alignment does not match whole-sentence tokenization: "
    + "; ".join(_equivalence_errors[:3]))
assert tokenizer_equivalence["tokenizer_equivalence_examples"] > 0, (
    "no example passed tokenizer equivalence (fail fast)")
print(json.dumps(tokenizer_equivalence, indent=2, sort_keys=True))


In [ ]:
alignment_preflight = {
    "aligned_example_count": 0,
    "unalignable_example_count": 0,
    "truncated_example_count": 0,
    "truncated_entity_count": 0,
    "fully_dropped_entity_count": 0,
    "partially_truncated_entity_count": 0,
    "supervised_token_count": 0,
    "positive_label_count": 0,
}
encoded_train = []
for row in train_examples:
    try:
        feature = encode_mention_example_slow(
            row,
            tokenizer,
            coverage_by_source=coverage_by_source,
            max_length=limits.max_sequence_length,
            segmented_text=segment_example_text(row["text"]),
        )
    except AlignmentError:
        alignment_preflight["unalignable_example_count"] += 1
        continue
    assert len(feature["input_ids"]) == len(feature["attention_mask"]) == len(feature["labels"]) == len(feature["label_mask"])
    alignment_preflight["aligned_example_count"] += 1
    alignment_preflight["truncated_example_count"] += int(bool(feature["truncated"]))
    alignment_preflight["truncated_entity_count"] += int(feature["truncated_entity_count"])
    alignment_preflight["fully_dropped_entity_count"] += int(feature["fully_dropped_entity_count"])
    alignment_preflight["partially_truncated_entity_count"] += int(feature["partially_truncated_entity_count"])
    alignment_preflight["supervised_token_count"] += sum(feature["label_mask"])
    alignment_preflight["positive_label_count"] += sum(1 for lab in feature["labels"] if any(lab))
    encoded_train.append(feature)

assert encoded_train, "alignment preflight produced no usable training features"
print(json.dumps(alignment_preflight, indent=2, sort_keys=True))


## Load the backbone (after alignment preflight passed)


In [ ]:
backbone = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
).to(device)
resolved_model_revision = getattr(backbone.config, "_commit_hash", "") or MODEL_REVISION
base_parameter_count_actual = sum(p.numel() for p in backbone.parameters())
print(json.dumps({
    "model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION,
    "resolved_model_revision": resolved_model_revision,
    "actual_base_parameters": base_parameter_count_actual,
    "registry_base_parameters": role.base_parameter_count,
    "cache_dir": str(MODEL_CACHE_DIR),
}, indent=2, sort_keys=True))


## 9. Dataset Encoding And Collation

In [ ]:
# encoded_train was built by the alignment preflight above (slow-tokenizer path).
encoded_train_repeat = []
for row in train_examples:
    try:
        encoded_train_repeat.append(encode_mention_example_slow(
            row, tokenizer, coverage_by_source=coverage_by_source,
            max_length=limits.max_sequence_length,
            segmented_text=segment_example_text(row["text"]),
        ))
    except AlignmentError:
        continue
encoded_validation = []
for row in validation_examples:
    try:
        encoded_validation.append(encode_mention_example_slow(
            row, tokenizer, coverage_by_source=coverage_by_source,
            max_length=limits.max_sequence_length,
            segmented_text=segment_example_text(row["text"]),
        ))
    except AlignmentError:
        continue
assert encoded_validation, "no alignable validation examples"
preparation_hash = preparation_digest(encoded_train)
assert preparation_hash == preparation_digest(encoded_train_repeat), "fixed-seed preparation is not deterministic"
batch = pad_encoded_features(
    encoded_train[:limits.batch_size],
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1,
)
batch_shapes = {
    "batch_size": len(batch["input_ids"]),
    "sequence_length": len(batch["input_ids"][0]),
    "entity_type_count": len(ENTITY_TYPE_ORDER),
    "preparation_hash": preparation_hash,
}
assert all(len(r) == batch_shapes["sequence_length"] for r in batch["attention_mask"])
assert all(len(r) == batch_shapes["sequence_length"] for r in batch["label_mask"])
print(json.dumps(batch_shapes, indent=2, sort_keys=True))


## 10. One Bounded Forward, Backward, And Optimizer Step

In [ ]:
class MentionTokenClassifier(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module, label_count: int) -> None:
        super().__init__()
        self.base_model = base_model
        hidden_size = int(base_model.config.hidden_size)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(hidden_size, label_count)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        output = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(output.last_hidden_state))

model = MentionTokenClassifier(backbone, len(ENTITY_TYPE_ORDER)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=limits.learning_rate)
criterion = torch.nn.BCEWithLogitsLoss(reduction="none")
trainable_parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
model.train()
input_ids = torch.tensor(batch["input_ids"], dtype=torch.long, device=device)
attention_mask = torch.tensor(batch["attention_mask"], dtype=torch.long, device=device)
labels = torch.tensor(batch["labels"], dtype=torch.float32, device=device)
label_mask = torch.tensor(batch["label_mask"], dtype=torch.float32, device=device)
optimizer.zero_grad(set_to_none=True)
logits = model(input_ids=input_ids, attention_mask=attention_mask)
loss_by_token = criterion(logits, labels)
normalizer = torch.clamp(label_mask.sum() * len(ENTITY_TYPE_ORDER), min=1.0)
loss = (loss_by_token * label_mask.unsqueeze(-1)).sum() / normalizer
assert torch.isfinite(loss).item(), "smoke loss is not finite"
loss.backward()
optimizer.step()
optimizer_step_confirmed = True
loss_values = {"train_loss": float(loss.detach().cpu()), "optimizer_step_confirmed": optimizer_step_confirmed}
print(json.dumps({
    "batch_shapes": batch_shapes,
    "loss_values": loss_values,
    "trainable_parameter_count": trainable_parameter_count,
}, indent=2, sort_keys=True))


## 11. Tiny Validation Inference

In [ ]:
model.eval()
validation_losses = []
positive_logits = 0
validation_batches_used = 0
with torch.no_grad():
    for start in range(0, min(len(encoded_validation), limits.batch_size * limits.max_validation_batches), limits.batch_size):
        features = encoded_validation[start:start + limits.batch_size]
        if not features:
            continue
        vb = pad_encoded_features(
            features,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0,
        )
        vi = torch.tensor(vb["input_ids"], dtype=torch.long, device=device)
        va = torch.tensor(vb["attention_mask"], dtype=torch.long, device=device)
        vl = torch.tensor(vb["labels"], dtype=torch.float32, device=device)
        vm = torch.tensor(vb["label_mask"], dtype=torch.float32, device=device)
        out = model(input_ids=vi, attention_mask=va)
        raw = criterion(out, vl)
        denom = torch.clamp(vm.sum() * len(ENTITY_TYPE_ORDER), min=1.0)
        vloss = (raw * vm.unsqueeze(-1)).sum() / denom
        assert torch.isfinite(vloss).item(), "validation smoke loss is not finite"
        validation_losses.append(float(vloss.cpu()))
        positive_logits += int(((torch.sigmoid(out) > 0.5) * vm.unsqueeze(-1)).sum().cpu())
        validation_batches_used += 1
validation_metrics = {
    "validation_batches": validation_batches_used,
    "validation_examples": min(len(encoded_validation), limits.batch_size * limits.max_validation_batches),
    "mean_validation_loss": sum(validation_losses) / max(1, len(validation_losses)),
    "positive_token_type_predictions": positive_logits,
}
print(json.dumps(validation_metrics, indent=2, sort_keys=True))


## 12. Checkpoint Save, Reload, And Manifest

In [ ]:
checkpoint_path = CHECKPOINT_DIR / "s1_mention_smoke_model.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "entity_type_order": ENTITY_TYPE_ORDER,
    "mode": "SMOKE_ONLY",
    "seed": SEED,
}, checkpoint_path)
checkpoint_sha256 = sha256_file(checkpoint_path)
loaded_checkpoint = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(loaded_checkpoint["model_state_dict"])
assert loaded_checkpoint["mode"] == "SMOKE_ONLY"
training_manifest = {
    "manifest_version": 1,
    "stage_id": "S1",
    "role": "mention/vihealthbert",
    "status": "SMOKE_ONLY",
    "smoke_only_not_full_training": True,
    "full_training_readiness": bool(
        PRODUCTION_SEGMENTATION
        and tokenizer_equivalence["tokenizer_equivalence_failures"] == 0
        and alignment_preflight["unalignable_example_count"] == 0
    ),
    "repository": {
        "repo_url": REPO_URL,
        "repo_ref": REPO_REF,
        "resolved_commit": RESOLVED_COMMIT,
    },
    "runtime": runtime_report,
    "corpus": corpus_report,
    "model": {
        **model_registry_report,
        "resolved_model_revision": resolved_model_revision,
        "actual_base_parameters": base_parameter_count_actual,
        "trainable_parameter_count": trainable_parameter_count,
    },
    "limits": {
        "max_train_examples": limits.max_train_examples,
        "max_validation_examples": limits.max_validation_examples,
        "max_train_batches": limits.max_train_batches,
        "max_validation_batches": limits.max_validation_batches,
        "max_optimizer_steps": limits.max_optimizer_steps,
        "batch_size": limits.batch_size,
        "max_sequence_length": limits.max_sequence_length,
        "learning_rate": limits.learning_rate,
    },
    "batch_shapes": batch_shapes,
    "tokenizer": {
        "tokenizer_class": tokenizer_report["tokenizer_class"],
        "tokenizer_is_fast": tokenizer_report["tokenizer_is_fast"],
        "tokenizer_revision": MODEL_REVISION,
        "vocab_size": tokenizer_report["vocab_size"],
    },
    "alignment": {
        "alignment_backend": ALIGNMENT_BACKEND,
        "alignment_backend_version": ALIGNMENT_BACKEND,
        "subtoken_supervision_policy": SUBTOKEN_SUPERVISION_POLICY,
        "aligned_example_count": alignment_preflight["aligned_example_count"],
        "unalignable_example_count": alignment_preflight["unalignable_example_count"],
        "truncated_example_count": alignment_preflight["truncated_example_count"],
        "truncated_entity_count": alignment_preflight["truncated_entity_count"],
        "fully_dropped_entity_count": alignment_preflight["fully_dropped_entity_count"],
        "partially_truncated_entity_count": alignment_preflight["partially_truncated_entity_count"],
        "partial_truncation_policy": PARTIAL_TRUNCATION_POLICY,
        **tokenizer_equivalence,
    },
    "word_segmentation": {
        "segmenter_mode": segmenter_report["segmenter_mode"],
        "degraded_fallback": segmenter_report["degraded_fallback"],
        "word_segmenter": segmenter_report["word_segmenter"],
        "word_segmenter_version": segmenter_report["word_segmenter_version"],
        "word_segmenter_resource_hashes": segmenter_report["word_segmenter_resource_hashes"],
        "acquisition_source": segmenter_report["acquisition_source"],
    },
    "loss_values": loss_values,
    "validation_metrics": validation_metrics,
    "loss_masks": {"vietmed_train_only": mask_report},
    "artifacts": {
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_sha256": checkpoint_sha256,
        "training_manifest_path": str(TRAINING_MANIFEST_PATH),
    },
}
TRAINING_MANIFEST_PATH.write_text(json.dumps(training_manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps({
    "status": "SMOKE_ONLY",
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_sha256": checkpoint_sha256,
    "training_manifest_path": str(TRAINING_MANIFEST_PATH),
    "checkpoint_reload_succeeded": True,
}, indent=2, sort_keys=True))


## 13. Return Artifacts

Return `OUTPUT_DIR` from Drive for review. It must contain `checkpoint/s1_mention_smoke_model.pt` and `training_manifest.json`. Do not return base-model cache files, run organizer inference, run packaging, or claim full S1 training.